# Escenario 3 — Sistema de Investigación Multi-Agente

**Dominios:** Agentic Architecture & Orchestration · Tool Design & MCP Integration · Context Management & Reliability

## Contexto
Construís un sistema de investigación donde un coordinador delega a subagentes especializados:
- Agente de búsqueda web
- Agente de análisis de documentos
- Agente de síntesis
- Agente de reportes

## Lo que vas a implementar
1. Coordinador con descomposición dinámica de tareas
2. Ejecución paralela de subagentes (múltiples Task calls en una respuesta)
3. Paso explícito de contexto con atribución de fuentes
4. Propagación de errores estructurada
5. Síntesis con manejo de datos contradictorios y brechas de cobertura

In [ ]:
import anthropic
import json
from typing import Any

client = anthropic.Anthropic()

## Paso 1 — Simular los subagentes especializados

**Concepto clave:** Los subagentes NO heredan contexto del coordinador. Todo el contexto necesario debe estar explícitamente en su prompt.

In [ ]:
def run_web_search_agent(query: str, focus_area: str) -> dict:
    """Subagente especializado en búsqueda web."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system="""Sos un investigador especializado en búsqueda web.
        Generá resultados de búsqueda simulados pero realistas sobre el tema dado.
        Incluye fuentes con URLs, fechas de publicación y extractos relevantes.
        Si el tema está fuera de tu área de especialización, indicalo claramente.""",
        messages=[{"role": "user", "content": f"""
Investigá: {query}
Área de enfoque: {focus_area}

Devuelve un JSON con esta estructura:
{{
  "success": true,
  "focus_area": "{focus_area}",
  "results": [
    {{
      "claim": "afirmación específica encontrada",
      "evidence_excerpt": "extracto relevante de la fuente",
      "source_url": "https://...",
      "source_name": "nombre de la publicación",
      "publication_date": "YYYY-MM-DD"
    }}
  ],
  "coverage_gaps": ["áreas no encontradas"]
}}
"""}]
    )
    
    try:
        text = response.content[0].text
        # Extraer JSON de la respuesta
        start = text.find('{')
        end = text.rfind('}') + 1
        return json.loads(text[start:end])
    except Exception:
        return {
            "success": False,
            "error_type": "parse_error",
            "focus_area": focus_area,
            "query_attempted": query,
            "partial_results": [],
            "alternatives": ["Reintentar con query más específica", "Usar fuentes alternativas"]
        }


def run_synthesis_agent(topic: str, web_results: list[dict], coverage_gaps: list[str]) -> dict:
    """Subagente de síntesis — recibe contexto EXPLÍCITO de los otros agentes."""
    
    # IMPORTANTE: pasar el contexto completo explícitamente
    context = json.dumps(web_results, indent=2, ensure_ascii=False)
    gaps = "\n".join(f"- {gap}" for gap in coverage_gaps) if coverage_gaps else "Ninguna"
    
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=2048,
        system="""Sos un investigador especializado en síntesis de información.
        Analizás múltiples fuentes y producís reportes estructurados.
        SIEMPRE preservás la atribución de fuentes.
        SIEMPRE anotás datos contradictorios en lugar de elegir arbitrariamente.
        SIEMPRE indicás las brechas de cobertura.""",
        messages=[{"role": "user", "content": f"""
Sintetizá la investigación sobre: {topic}

RESULTADOS DE BÚSQUEDA WEB:
{context}

BRECHAS DE COBERTURA CONOCIDAS:
{gaps}

Producí un reporte con secciones:
1. Hallazgos bien establecidos (múltiples fuentes concordantes)
2. Hallazgos con evidencia mixta (fuentes contradictorias — anotar ambas con atribución)
3. Brechas de cobertura (qué no se pudo investigar y por qué)
4. Conclusiones y recomendaciones

Para datos contradictorios, usar formato:
"[Fuente A, fecha]: valor X | [Fuente B, fecha]: valor Y — Nota metodológica: ..."
"""}]
    )
    
    return {
        "success": True,
        "synthesis": response.content[0].text,
        "sources_preserved": True
    }

## Paso 2 — Coordinador con descomposición dinámica

**Concepto clave:** El coordinador analiza la consulta y decide dinámicamente qué subagentes invocar. No siempre pasa por el pipeline completo.

In [ ]:
def research_coordinator(topic: str) -> str:
    """Coordinador que descompone la investigación y ejecuta subagentes en paralelo."""
    
    print(f"\n📋 INVESTIGANDO: {topic}")
    print("=" * 60)
    
    # Paso 1: El coordinador descompone el tema en áreas de enfoque
    decompose_response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        messages=[{"role": "user", "content": f"""
Descomponé el tema de investigación '{topic}' en 3-4 áreas de enfoque DISTINTAS
que cubran el espacio completo del problema sin solapamiento.

Devuelve SOLO un array JSON: ["área 1", "área 2", "área 3"]
Asegurate de cubrir TODAS las dimensiones relevantes del tema.
"""}]
    )
    
    text = decompose_response.content[0].text
    start = text.find('[')
    end = text.rfind(']') + 1
    focus_areas = json.loads(text[start:end])
    
    print(f"\n🔍 Áreas identificadas para investigar en PARALELO:")
    for area in focus_areas:
        print(f"  • {area}")
    
    # Paso 2: Ejecutar subagentes en paralelo (en producción serían Tasks simultáneas)
    # Aquí los ejecutamos secuencialmente pero la arquitectura es la misma
    print(f"\n⚡ Ejecutando {len(focus_areas)} subagentes...")
    
    all_results = []
    all_gaps = []
    
    for area in focus_areas:
        print(f"  → Agente de búsqueda: {area}")
        result = run_web_search_agent(topic, area)
        
        if result.get("success"):
            all_results.extend(result.get("results", []))
            all_gaps.extend(result.get("coverage_gaps", []))
        else:
            # Propagación de error estructurada
            print(f"  ⚠️ Error en subagente para '{area}': {result.get('error_type')}")
            all_gaps.append(f"{area}: {result.get('error_type', 'fallo desconocido')}")
    
    print(f"\n  ✓ {len(all_results)} hallazgos recopilados")
    print(f"  ⚠️ {len(all_gaps)} brechas de cobertura identificadas")
    
    # Paso 3: Síntesis con contexto completo pasado explícitamente
    print(f"\n📊 Sintetizando hallazgos...")
    synthesis = run_synthesis_agent(topic, all_results, all_gaps)
    
    return synthesis["synthesis"]


# Ejecutar investigación
report = research_coordinator("impacto de la IA en las industrias creativas")
print("\n" + "=" * 60)
print("REPORTE FINAL:")
print("=" * 60)
print(report)

## Paso 3 — Análisis del Escenario del Examen

**Pregunta del examen:** El coordinador descompone "impacto de la IA en las industrias creativas" en:
- "IA en arte digital"
- "IA en diseño gráfico"
- "IA en fotografía"

¿Cuál es el problema?

In [ ]:
# DIAGNÓSTICO: descomposición demasiado estrecha

print("PROBLEMA IDENTIFICADO EN LOGS DEL COORDINADOR:")
print()
print("Subtareas asignadas:")
bad_decomposition = ["IA en arte digital", "IA en diseño gráfico", "IA en fotografía"]
for area in bad_decomposition:
    print(f"  ✗ {area}  ← todas son artes VISUALES")

print()
print("Áreas omitidas:")
missing = ["IA en música", "IA en escritura/literatura", "IA en producción cinematográfica",
           "IA en videojuegos", "IA en danza/performance"]
for area in missing:
    print(f"  ✗ {area}  ← no investigado")

print()
print("CAUSA RAÍZ: La descomposición del coordinador es demasiado estrecha.")
print("Los subagentes ejecutaron CORRECTAMENTE las subtareas asignadas.")
print("El problema no está en la búsqueda web, el análisis o la síntesis.")
print("El problema está en LO QUE SE LES ASIGNÓ investigar.")

print()
print("RESPUESTA CORRECTA: B")
print("'La descomposición de tareas del coordinador es demasiado estrecha,")
print(" resultando en asignaciones que no cubren todos los dominios relevantes.'")

## Paso 4 — Propagación de errores estructurada

**Pregunta del examen:** El subagente de búsqueda web se agota por timeout. ¿Qué debe devolver?

In [ ]:
# Comparar los 4 enfoques de la pregunta del examen

print("Opción A — Error estructurado (CORRECTO):")
option_a = {
    "success": False,
    "error_type": "timeout",
    "query_attempted": "impacto IA en música 2024",
    "partial_results": [{"claim": "hallazgo parcial antes del timeout"}],
    "alternatives": [
        "Reintentar con query más específica",
        "Usar fuente alternativa: Google Scholar",
        "Continuar sin este tema y anotar la brecha"
    ]
}
print(json.dumps(option_a, indent=2, ensure_ascii=False))
print("→ El coordinador puede tomar decisiones informadas de recuperación")

print()
print("Opción B — Estado genérico (MALO):")
option_b = {"status": "search unavailable"}
print(json.dumps(option_b, indent=2))
print("→ El coordinador NO sabe si fue un timeout, si hay resultados parciales,")
print("  o si puede reintentar con otra estrategia")

print()
print("Opción C — Resultado vacío como éxito (MALO):")
option_c = {"success": True, "results": []}
print(json.dumps(option_c, indent=2))
print("→ El coordinador cree que no hay datos disponibles, no que hubo un error")
print("  El reporte final tendrá brechas sin anotar")

print()
print("Opción D — Excepción propagada (MALO):")
print("raise TimeoutException('Web search timeout')")
print("→ Termina TODO el flujo de trabajo de investigación")
print("  Incluso si los otros subagentes tuvieron éxito")

## Preguntas de Reflexión

1. ¿Por qué los subagentes no pueden acceder al historial de conversación del coordinador?
2. Si el coordinador emite 3 Tasks en una sola respuesta, ¿eso significa ejecución paralela o secuencial?
3. ¿Qué debe incluir el reporte final cuando dos fuentes dan estadísticas contradictorias?
4. ¿Cuándo el coordinador debería reintentar un subagente vs. continuar con una brecha anotada?
5. ¿Por qué la opción A del examen (herramienta verify_fact con alcance limitado para el agente de síntesis) reduce la latencia sin violar el principio de mínimo privilegio?